# Run the EEG pipeline

Run the cells in order. Select the intended Python environment as the notebook kernel: the installation cell modifies that environment. Start Jupyter in a Valeria GPU session after loading the Python and CUDA modules and activating the project environment, as described in the README. Loading modules in a notebook cell does not configure an already running kernel.

The repository and EEG data must already be present on the machine. The pinned requirements were recorded on Valeria and are not guaranteed to install on Colab or other platforms. Installation requires access to the configured Valeria wheelhouse and PyPI. The pipeline can take many hours; keep the session active.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Change this if the notebook was not started from the repository root.
repo = Path.cwd().resolve()
pipeline_args = ["--workers", "1"]  # Add "--rapide" only for a reduced test.

for name in ("requirement.txt", "00_pipeline.py"):
    if not (repo / name).is_file():
        raise FileNotFoundError(f"{name} not found in {repo}; set repo to the project root.")

print(f"Repository: {repo}")
print(f"Python kernel: {sys.executable}")

## Install and check requirements

This uses the Valeria installation command documented in the README. It can fail if a pinned wheel or dependency is unavailable. Restart the kernel if you already imported project dependencies before installing them.

In [ ]:
def run_command(command):
    print("Running:", subprocess.list2cmdline([str(arg) for arg in command]), flush=True)
    with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

run_command([sys.executable, "-m", "pip", "--isolated", "install", "--only-binary=:all:",
             "--index-url", "https://pypi.org/simple", "-r", str(repo / "requirement.txt")])
run_command([sys.executable, "-m", "pip", "check"])

## Run the pipeline

The launcher detects available GPU backends and reports CPU fallbacks. Results and logs are saved under `sorties`. A failed command stops the cell and reports its exit code.

In [ ]:
run_command([sys.executable, "-u", str(repo / "00_pipeline.py"), *pipeline_args])